# 00 — Environment Test
## Inflation Prediction and Economic Signal Analysis

**Purpose:** Verify that all dependencies are installed correctly and FRED API connection works.

**Expected outcome:**
- All imports succeed
- FRED API returns data for a test series (US CPI)
- Basic plot renders correctly

**Run this before starting Phase 1.**

---
## 1. Import Test

In [ ]:
# --- Standard Libraries ---
import os
import sys

# --- Data ---
import pandas as pd
import numpy as np

# --- FRED API ---
from fredapi import Fred
from dotenv import load_dotenv

# --- Econometrics ---
import statsmodels
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.vector_ar.var_model import VAR

# --- Machine Learning ---
import sklearn
from sklearn.linear_model import Ridge

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns

print('='*50)
print('All imports successful')
print('='*50)
print(f'Python      : {sys.version[:6]}')
print(f'pandas      : {pd.__version__}')
print(f'numpy       : {np.__version__}')
print(f'statsmodels : {statsmodels.__version__}')
print(f'scikit-learn: {sklearn.__version__}')

---
## 2. API Key Setup Test

In [ ]:
# Load API key from .env file
load_dotenv()

api_key = os.getenv('FRED_API_KEY')

if api_key is None or api_key == 'your_fred_api_key_here':
    print('ERROR: FRED_API_KEY not set.')
    print('Steps:')
    print('  1. Copy .env.example to .env')
    print('  2. Replace placeholder with your key from:')
    print('     https://fred.stlouisfed.org/docs/api/api_key.html')
else:
    print('API key loaded successfully')
    print(f'Key preview: {api_key[:6]}...')

---
## 3. FRED API Connection Test

In [ ]:
# Initialise FRED client
fred = Fred(api_key=api_key)

# Test fetch: US CPI (CPIAUCSL)
# Monthly, seasonally adjusted, 2000-01-01 to present
try:
    cpi_us = fred.get_series(
        'CPIAUCSL',
        observation_start='2000-01-01'
    )
    print('FRED API connection: SUCCESS')
    print(f'Series: CPIAUCSL (US CPI)')
    print(f'Observations: {len(cpi_us)}')
    print(f'Date range: {cpi_us.index[0].date()} to {cpi_us.index[-1].date()}')
    print(f'Latest value: {cpi_us.iloc[-1]:.2f}')
except Exception as e:
    print(f'FRED API connection FAILED: {e}')

---
## 4. Quick Data Preview

In [ ]:
# Convert to DataFrame and preview
df_test = cpi_us.to_frame(name='CPI_US')
df_test.index.name = 'date'

print('Shape:', df_test.shape)
print()
print('First 5 rows:')
print(df_test.head())
print()
print('Last 5 rows:')
print(df_test.tail())
print()
print('Null values:', df_test.isnull().sum().values[0])

---
## 5. Test Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df_test.index, df_test['CPI_US'], color='steelblue', linewidth=1.5)

# Mark structural break points
breaks = {
    '2008-09': 'GFC',
    '2020-03': 'COVID',
    '2022-02': 'Energy Shock'
}
for date_str, label in breaks.items():
    ax.axvline(
        pd.Timestamp(date_str),
        color='red',
        linestyle='--',
        alpha=0.6,
        linewidth=1
    )
    ax.text(
        pd.Timestamp(date_str),
        df_test['CPI_US'].max() * 0.97,
        f' {label}',
        color='red',
        fontsize=8
    )

ax.set_title('US CPI (2000–Present) — Environment Test', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('CPI Index')
ax.grid(True, alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()

print('Plot rendered successfully')

---
## 6. Environment Summary

In [ ]:
print('='*50)
print('ENVIRONMENT TEST SUMMARY')
print('='*50)
print()
checks = [
    ('All imports loaded',   True),
    ('FRED API key found',   api_key is not None and api_key != 'your_fred_api_key_here'),
    ('FRED API connected',   len(cpi_us) > 0),
    ('DataFrame created',    len(df_test) > 0),
    ('Plot rendered',        True),
]
all_passed = True
for label, status in checks:
    icon = '✅' if status else '❌'
    print(f'  {icon}  {label}')
    if not status:
        all_passed = False

print()
if all_passed:
    print('All checks passed. Ready to proceed to Phase 1.')
else:
    print('Some checks failed. Resolve issues before Phase 1.')

---
## Next Step

If all checks passed → proceed to `01_data_collection.ipynb`